# Red Wine Quality Prediction

**Goal:** Predict wine quality score (0-10) and classify Good vs Bad
**Algorithm:** Random Forest (both Regressor & Classifier)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
%matplotlib inline

In [ ]:
np.random.seed(42)
n = 1000

fixed_acidity    = np.random.uniform(4, 16, n).round(2)
volatile_acidity = np.random.uniform(0.1, 1.6, n).round(3)
citric_acid      = np.random.uniform(0, 1, n).round(2)
residual_sugar   = np.random.uniform(0.5, 15, n).round(2)
chlorides        = np.random.uniform(0.01, 0.6, n).round(3)
free_sulfur      = np.random.uniform(1, 70, n).round(1)
total_sulfur     = free_sulfur + np.random.uniform(5, 150, n).round(1)
density          = np.random.uniform(0.99, 1.005, n).round(4)
pH               = np.random.uniform(2.8, 4, n).round(2)
sulphates        = np.random.uniform(0.3, 2, n).round(2)
alcohol          = np.random.uniform(8, 15, n).round(1)

quality = (3 + (alcohol * 0.3) + (fixed_acidity * 0.1)
           - (volatile_acidity * 1.5) + (sulphates * 1.0)
           + np.random.normal(0, 1, n)).astype(int)
quality = np.clip(quality, 3, 8)

df = pd.DataFrame({
    'fixed_acidity': fixed_acidity,
    'volatile_acidity': volatile_acidity,
    'citric_acid': citric_acid,
    'residual_sugar': residual_sugar,
    'chlorides': chlorides,
    'free_sulfur_dioxide': free_sulfur,
    'total_sulfur_dioxide': total_sulfur,
    'density': density,
    'pH': pH,
    'sulphates': sulphates,
    'alcohol': alcohol,
    'quality': quality
})
print ('Shape: %s' % (df.shape,))
print ('Columns: %s' % list(df.columns))

<hr>## 1. Exploratory Data Analysis

In [ ]:
print ('Quality distribution:\n%s' % df['quality'].value_counts().sort_index())
print ('\nStatistical summary:\n%s' % df.describe())
print ('\nMissing values: %s' % df.isnull().sum().sum())

<hr>## 2. Prepare Features & Target

In [ ]:
X = df.drop('quality', axis=1)
y_reg = df['quality']
y_clf = (df['quality'] >= 7).astype(int)

print ('Good wines (>=7): %d out of %d' % (y_clf.sum(), len(y_clf)))

<hr>## 3. Train Models

In [ ]:
X_train, X_test, yr_train, yr_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)
_, _, yc_train, yc_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42)

In [ ]:
reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_train, yr_train)
print ('Regressor: %s' % reg)

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, yc_train)
print ('Classifier: %s' % clf)

<hr>## 4. Evaluate Regression

In [ ]:
yr_pred = reg.predict(X_test)
print ('Regression Results (predict exact score):')
print ('RMSE: %.3f' % np.sqrt(mean_squared_error(yr_test, yr_pred)))
print ('R²:   %.3f' % r2_score(yr_test, yr_pred))

<hr>## 5. Evaluate Classification

In [ ]:
yc_pred = clf.predict(X_test)
print ('Classification Results (Good [>=7] vs Bad):')
print ('Accuracy: %.4f' % accuracy_score(yc_test, yc_pred))
print ('\nClassification Report:\n%s' % classification_report(
    yc_test, yc_pred, target_names=['Bad', 'Good']))

<hr>## 6. Feature Importance

In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': reg.feature_importances_
}).sort_values('importance', ascending=False)
print ('Top 5 features:\n%s' % importances.head(5).to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(importances['feature'], importances['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance for Wine Quality')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()